# فصل 4: توابع و سازمان‌دهی کد

کد این نوت‌بوک عیناً از فصل 4 کتاب «مهندسی هوش مصنوعی مکانی» برداشته شده است.

پیش از اجرا: مسیر داده‌ها را با ساختار پوشه‌ی `data/`، `work/` و `out/` هماهنگ کنید. راه دانلود هر فایل در پیوست ب کتاب آمده است.


## ۴-۲) تعریف تابع با def


In [ ]:
def print_hospital_info(hospital):
    """Nicely print one hospital's information."""
    print("-" * 40)
    print(f"Name: {hospital['name']}")
    print(f"Capacity: {hospital['capacity']} beds")
    print(f"Location: ({hospital['lat']}, {hospital['lon']})")
milad = {"name": "Milad", "capacity": 1000, "lat": 35.7350, "lon": 51.3650}
print_hospital_info(milad)   # this line calls the function

## ۴-۳) پارامتر و آرگومان


In [ ]:
# city_name and population are PARAMETERS
def show_city(city_name, population):
    print(f"{city_name} has {population:,} people")

# "Tehran" and 8693706 are ARGUMENTS
show_city("Tehran", 8693706)
show_city("Isfahan", 1961260)

## ۴-۴) مقدار بازگشتی: return


In [ ]:
from math import radians, cos

def approx_distance(lat1, lon1, lat2, lon2):
    mean_lat = radians((lat1 + lat2) / 2)
    d = (((lat1 - lat2) * 111) ** 2
         + ((lon1 - lon2) * 111 * cos(mean_lat)) ** 2)
    return d ** 0.5

result = approx_distance(35.7, 51.3, 35.8, 51.4)
print(f"{result:.2f} km")
print(f"{result * 1000:.0f} m")   # we can reuse the returned value

## ۴-۵) پارامترهای پیش‌فرض، فرمول هاورساین و آرگومان‌های منعطف


In [ ]:
from math import radians, sin, cos, sqrt, asin
def haversine(lat1, lon1, lat2, lon2, earth_radius=6371):
    """Great-circle distance in km (earth_radius defaults to 6371)."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return earth_radius * 2 * asin(sqrt(a))
print(haversine(35.6892, 51.3890, 32.6539, 51.6660))   # Tehran -> Isfahan
print(haversine(0, 0, 0, 90, earth_radius=1737))       # on the Moon!

## ۴-۶) محدوده‌ی متغیرها


In [ ]:
city_count = 5                # a GLOBAL variable

def analyze():
    average_pop = 2.5         # a LOCAL variable
    print(f"Analyzing {city_count} cities")   # reads the global: fine

analyze()
print(city_count)             # works: it is global
print(average_pop)            # NameError: it is local to analyze()

## ۴-۷) ماژول‌ها و دستور import


In [ ]:
import geo_utils                  # the whole module
d = geo_utils.haversine(35.7, 51.4, 32.6, 51.7)
from geo_utils import haversine   # just one function
d = haversine(35.7, 51.4, 32.6, 51.7)
import numpy as np                # with an alias (very common)

## ۴-۸) پروژه‌ی پایان فصل: ساخت جعبه‌ابزار geo_utils


In [ ]:
"""GeoAI Toolkit - basic helpers for spatial analysis. v1.0"""
from math import radians, sin, cos, sqrt, asin

EARTH_RADIUS_KM = 6371

def validate_coordinates(lat, lon):
    """Return True if the coordinate is within valid ranges."""
    return -90 <= lat <= 90 and -180 <= lon <= 180

def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in km; raises on invalid input."""
    if not (validate_coordinates(lat1, lon1)
            and validate_coordinates(lat2, lon2)):
        raise ValueError("invalid coordinates")
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return EARTH_RADIUS_KM * 2 * asin(sqrt(a))

def create_buffer_box(lat, lon, radius_km):
    """Return the bounding box around a point.

    North-south is a constant 111 km per degree, but east-west shrinks
    towards the poles, so the longitude step is divided by cos(lat).
    """
    d_lat = radius_km / 111
    d_lon = radius_km / (111 * cos(radians(lat)))
    return {
        "north": lat + d_lat,
        "south": lat - d_lat,
        "east": lon + d_lon,
        "west": lon - d_lon,
    }

def find_nearest(target, candidates, top_n=3):
    """Return the top_n nearest candidates to the target point."""
    out = []
    for c in candidates:
        try:
            d = haversine(target["lat"], target["lon"], c["lat"], c["lon"])
            out.append({**c, "distance": d})
        except (ValueError, KeyError):
            continue   # skip dirty records
    out.sort(key=lambda x: x["distance"])
    return out[:top_n]

In [ ]:
from geo_utils import haversine, find_nearest
print(f"{haversine(35.6892, 51.3890, 32.6539, 51.6660):.2f} km")
crisis = {"lat": 35.72, "lon": 51.38}
hospitals = [
    {"name": "Milad", "lat": 35.735, "lon": 51.365},
    {"name": "Shariati", "lat": 35.740, "lon": 51.420},
    {"name": "Heart", "lat": 35.710, "lon": 51.400},
]
for h in find_nearest(crisis, hospitals, top_n=2):
    print(f"{h['name']}: {h['distance']:.2f} km")